# 03. Enigma + Stokastisk build 5: OCI runtime integration

**Accelerator: GPU T4 x2** · fresh Kaggle session · Python 3.13 · generated by `scripts/build_kaggle_diagnostic_notebooks.py`

**Prepared; NOT yet executed end to end (no GPU or Playground acceptance exists for this notebook).** One integration mode, `oci-production`: the Enigma API on the OCI Always Free deployment, reached by outbound HTTPS only (no tunnel; the local vLLM server stays on loopback). BLOCKED before any request or GPU work while the deployment does not exist. Order: GPU environment guard; both credential preflights; a durable diagnostic acknowledgement; the historical baseline manifest (notebooks 01 + 02, digest-verified); the capability gate (HTTP, schema, deployment, durable evidence: four separate results); real runtime authentication (read-only); a 90 s post-capability survival gate; then immutable build 5, real local Qwen inference, the reviewed connector in its own supervised subprocess with explicit intervals, offline NVML and vLLM stages, registration, heartbeat, idle job polling, single and continuous telemetry, the full connector with one real Playground job, build-5-safe shutdown and an OVERALL verdict (PASS / FAIL / INCONCLUSIVE / BLOCKED).

**Kaggle Secrets (Add-ons → Secrets; tick each one for this notebook).** `ENIGMA_DIAG_KEY` = the 56-character diagnostic key (`edx_…`, issued once by `scripts/diag_keys.py create` on the OCI host); it authenticates only `POST /api/v1/diag/events`. `ENIGMA_RUNTIME_TOKEN` = the 60-character Enigma runtime key (`enig_rt_…`, issued in the Enigma web app); it authenticates only runtime traffic. They are never interchangeable; no cloud-provider credential (OCI, AWS) is ever needed in Kaggle. If a credential preflight cell reports a problem, fix the secret in the sidebar and **re-run only that cell** (and the cell after it): the attempt id, event sequence and earlier verdicts are kept. A new Kaggle session always starts from the top.

**Evidence durability.** Every step prints flushed `MARK` and `VERDICT` lines and appends them to an fsynced `markers.jsonl` in a new, never-reused attempt directory. After a complete platform termination the VM's files may be gone (on 2026-11-26 a new session had to re-clone everything), but cell output printed before the failure was kept in the saved notebook. **Missing files do not prove that no crash occurred.** After a fatal session, copy the last `MARK` lines (UTC time, `t` = seconds since the prelude) into the incident report.

Run the cells top to bottom (or *Run all*). Every step ends with `VERDICT <step>: PASS | FAIL | INCONCLUSIVE`; anything but PASS raises and stops the run. Then run the final **evidence export** cell manually.

In [ ]:
# P0. Prelude: environment guard, attempt id, markers and verdicts. Standard library only: no network, no GPU library is imported.
import os, sys, json, time, uuid, platform, shutil, subprocess, re, pathlib
NOTEBOOK = "03_enigma_stokastisk_build5_oci_integration"
EXPECT = "gpu"                                   # "cpu" = Accelerator None; "gpu" = GPU T4 x2
assert os.path.isdir("/kaggle/working"), "Kaggle only: refusing to run anywhere else (never on a developer laptop)"


class DiagnosticStop(Exception):
    """Raised by every non-PASS verdict so that 'Run all' stops here. Later cells must never run after a failure."""


HAS_GPU_DEVICE = os.path.exists("/dev/nvidia0")
if EXPECT == "cpu" and HAS_GPU_DEVICE:
    raise DiagnosticStop("INCONCLUSIVE: wrong environment. A GPU is attached; set Accelerator = None and start a NEW session")
if EXPECT == "gpu" and not HAS_GPU_DEVICE:
    raise DiagnosticStop("INCONCLUSIVE: wrong environment. No GPU; set Accelerator = GPU T4 x2 and start a NEW session")
assert sys.version_info[:2] == (3, 13), f"expected Python 3.13 (build-5 wheel is cp313), got {platform.python_version()}"
REQUIRE_NO_INPUTS = False                       # notebook 01 must run with NO attached dataset, competition or utility script
_inputs = sorted(os.listdir("/kaggle/input")) if os.path.isdir("/kaggle/input") else []
if REQUIRE_NO_INPUTS and _inputs:
    raise DiagnosticStop(f"INCONCLUSIVE: wrong environment. Inputs are attached {_inputs}; remove every input and start a NEW session")
# ---- configuration: edit ONLY these lines, before running anything; they are fixed for the whole attempt ----
VALIDATION_MODE = "oci-required"
INTEGRATION_MODE = "oci-production"   # the OCI Always Free deployment (the only cloud endpoint); BLOCKED while OCI_BASE_URL is empty
RUNTIME_TRANSPORT = "oci"            # fixed for the attempt, never switched, no fallback
# ------------------------------------------------------------------------------------------------------------
assert VALIDATION_MODE in ("standalone", "oci-assisted", "oci-required"), VALIDATION_MODE
assert RUNTIME_TRANSPORT in ("none", "oci"), RUNTIME_TRANSPORT
EXPERIMENT = ("build5" if EXPECT == "gpu" else "no-sdk") + ("" if VALIDATION_MODE == "oci-required" else "-" + VALIDATION_MODE) \
    + ("" if RUNTIME_TRANSPORT == "none" else "-via-" + RUNTIME_TRANSPORT)   # in every MARK

ATTEMPT = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "-" + uuid.uuid4().hex[:8]
ATTEMPT_DIR = f"/kaggle/working/enigma-diag-attempts/{NOTEBOOK}/{ATTEMPT}"
os.makedirs(ATTEMPT_DIR, exist_ok=False)               # a new directory per attempt: earlier evidence is never overwritten
T0 = time.monotonic()
RESULTS = {}                                           # verdicts of THIS kernel session only (stale outputs can never satisfy a check)

# Evidence redaction (shared; generated from scripts/kaggle_cells/evidence.py). Defence in depth: credentials are never recorded on purpose, and every
# MARK line and every exported text file is still scrubbed of anything shaped like an Enigma runtime key, a diagnostic key, an AWS key id, a bearer
# token or a JWT before it is written or archived.
import re, pathlib, zipfile
_SECRET_RE = re.compile(r"(?:enig_rt_|edx_)[a-z0-9]{8}_[A-Za-z0-9_-]{10,}|\b(?:AKIA|ASIA)[0-9A-Z]{16}\b|(?i:bearer)\s+[A-Za-z0-9._~+/=-]{8,}"
                        r"|\beyJ[A-Za-z0-9_-]{6,}\.[A-Za-z0-9_-]{6,}\.[A-Za-z0-9_-]*")
_EXPORT_SKIP_DIRS = ("enigma-src",)                    # the public connector clone is not evidence
_EXPORT_SKIP_SUFFIXES = (".whl",)                      # the public 38 MB wheel is not evidence (its sha256 is in the markers)


def redact(text):
    return _SECRET_RE.sub("<REDACTED>", text)


def write_redacted_archive(src_dir, zip_path):
    """Zip src_dir, scrubbing every UTF-8 text file. Returns (files, redactions)."""
    files = hits = 0
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(pathlib.Path(src_dir).rglob("*")):
            rel = p.relative_to(src_dir)
            if not p.is_file() or p.is_symlink() or rel.parts[0] in _EXPORT_SKIP_DIRS or p.suffix in _EXPORT_SKIP_SUFFIXES:
                continue
            data = p.read_bytes()
            try:
                text = data.decode("utf-8")
            except UnicodeDecodeError:
                z.writestr(str(rel), data)
            else:
                hits += len(_SECRET_RE.findall(text))
                z.writestr(str(rel), redact(text))
            files += 1
    return files, hits


def _read(path):
    try:
        with open(path) as fh:
            return fh.read(65536)
    except OSError:
        return None


def host():
    """Host memory, cgroup v2 memory (current, max, events), shared memory, disk, uptime and load. No environment variables are read."""
    h = {}
    for line in (_read("/proc/meminfo") or "").splitlines():
        k, _, v = line.partition(":")
        if k in ("MemTotal", "MemAvailable", "Shmem"):
            h[k.lower() + "_mib"] = round(int(v.split()[0]) / 1024, 1)
    mib = lambda s: None if s is None or not s.strip().isdigit() else round(int(s) / 2**20, 1)   # noqa: E731 ("max" = unlimited -> None)
    h["cgroup_current_mib"], h["cgroup_max_mib"] = mib(_read("/sys/fs/cgroup/memory.current")), mib(_read("/sys/fs/cgroup/memory.max"))
    h["cgroup_events"] = {k: int(v) for k, _, v in (l.partition(" ") for l in (_read("/sys/fs/cgroup/memory.events") or "").splitlines()) if v.strip().isdigit()}
    for d in ("/kaggle/working", "/tmp", "/dev/shm"):
        try:
            u = shutil.disk_usage(d)
            h["disk_" + d.strip("/").replace("/", "_")] = {"used_gib": round(u.used / 2**30, 2), "free_gib": round(u.free / 2**30, 2)}
        except OSError:
            pass
    up = _read("/proc/uptime")
    h["vm_uptime_s"] = float(up.split()[0]) if up else None
    h["loadavg"] = os.getloadavg()
    return h


def ident():
    return {"pid": os.getpid(), "ppid": os.getppid(), "pgid": os.getpgid(0), "sid": os.getsid(0)}


def mark(name, **fields):
    rec = {"attempt": ATTEMPT, "nb": NOTEBOOK, "experiment": EXPERIMENT, "mode": VALIDATION_MODE, "transport": RUNTIME_TRANSPORT, "marker": name, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "t": round(time.monotonic() - T0, 2), **ident(), **host(), **fields}
    line = redact(json.dumps(rec, default=str))
    with open(f"{ATTEMPT_DIR}/markers.jsonl", "a") as fh:
        fh.write(line + "\n"); fh.flush(); os.fsync(fh.fileno())
    print("MARK", line, flush=True)
    return rec


def verdict(step, status, reason, **evidence):
    assert status in ("PASS", "FAIL", "INCONCLUSIVE"), status
    RESULTS[step] = {"status": status, "reason": reason, "attempt": ATTEMPT, **evidence}
    mark("verdict", step=step, status=status, reason=reason, **evidence)
    print(f"VERDICT {step}: {status} | {reason}", flush=True)
    if status != "PASS":
        raise DiagnosticStop(f"{step} {status}: {reason}")


def require(*steps):
    """Every prerequisite must have PASSED in this very session (same attempt id)."""
    bad = [s for s in steps if RESULTS.get(s, {}).get("status") != "PASS" or RESULTS[s].get("attempt") != ATTEMPT]
    if bad:
        raise DiagnosticStop(f"INCONCLUSIVE: prerequisite(s) did not PASS in this session: {bad}. Start a new session and run from the top.")


# Survival observation (shared; generated from scripts/kaggle_cells/observe.py). PASSIVE by default: local, fsynced, printed MARK lines only (no
# network at all), with explicit sleep-begin / sleep-end markers, so the saved notebook output brackets a platform termination to within one tick.
# Remote diagnostic markers (OCI deployment) are OPT-IN at a few explicit offsets (bounded: at most 12 per window), each bracketed by local remote-submit / remote-ack markers.
# A missing cgroup OOM counter is reported as "unavailable", never as 0.
def oom_kill_count():
    ev = host()["cgroup_events"]
    return ev.get("oom_kill") if "oom_kill" in ev else None


def tick_schedule(seconds, every, dense_first_s=0.0, dense_every=2.0):
    """Monotonic offsets (s) at which a passive tick is recorded: every `dense_every` s for the first `dense_first_s` s, then every `every` s."""
    out, t = [], 0.0
    while True:
        step = dense_every if t < dense_first_s else every
        t = round(min(seconds, t + step), 3)
        out.append(t)
        if t >= seconds:
            return out


def observe(step, seconds, every=15.0, dense_first_s=0.0, dense_every=2.0, remote_at=(), sample=None, durable_begin=False):
    """Survival window. remote_at: offsets (s) where ONE bounded diagnostic marker is sent (active sampling); () = fully passive.
    durable_begin: the window starts only after the OCI deployment acknowledged an observation-start marker. PASS = full duration elapsed, no OOM kill observed,
    no failed sample, and every requested remote marker acknowledged."""
    remote_at = sorted(float(x) for x in remote_at)
    assert len(remote_at) <= 12, "remote markers per window are bounded"
    oom0 = oom_kill_count()
    if durable_begin:
        rcheck(step, step + "_obs_begin", stage=step, phase="before", note=f"window {seconds}s remote_at={remote_at}")
    start, problems, sent, n = time.monotonic(), [], [], 0
    mark(step + "_begin", seconds=seconds, every=every, dense_first_s=dense_first_s, dense_every=dense_every, remote_at=remote_at,
         oom_state="available" if oom0 is not None else "unavailable")
    for target in tick_schedule(seconds, every, dense_first_s, dense_every):
        n += 1
        mark(step + "_sleep_begin", i=n, until_s=target)
        time.sleep(max(0.0, target - (time.monotonic() - start)))
        el = round(time.monotonic() - start, 2)
        mark(step + "_sleep_end", i=n, elapsed_s=el)
        extra = sample() if sample else {}
        if extra.get("problem"):
            problems.append(extra["problem"])
        while remote_at and el + 1e-6 >= remote_at[0]:
            off = remote_at.pop(0)
            mark(step + "_remote_submit", offset_s=off)
            res = rmark(step + "_alive", stage=step, phase="observe", offset_s=off)
            mark(step + "_remote_result", offset_s=off, code=res.get("code"), acknowledged=bool(res))
            sent.append(bool(res))
            if not res and res.get("code") != "REMOTE_DISABLED":
                problems.append(f"remote marker at {off}s not acknowledged ({res.get('code')})")
        if extra:
            mark(step + "_sample", i=n, **extra)
    el, oom1 = time.monotonic() - start, oom_kill_count()
    delta = "unavailable" if oom0 is None or oom1 is None else oom1 - oom0
    ok = el >= seconds - 1.0 and delta in (0, "unavailable") and not problems
    mark(step + "_complete", observed_s=round(el, 2), ticks=n, remote_sent=len(sent), remote_acknowledged=sum(sent))
    verdict(step, "PASS" if ok else "FAIL", f"observed {el:.0f}/{seconds}s, oom_kill delta {delta}, remote {sum(sent)}/{len(sent)} acknowledged, "
            f"problems {problems[:3]}", observed_s=round(el, 1), oom_kill_delta=delta, ticks=n, remote_sent=len(sent), remote_acknowledged=sum(sent))


def export_evidence():
    """Zip this attempt's directory (only it), redacting every text file. Safe to run after a failure."""
    path = f"/kaggle/working/enigma-diag-attempts/{NOTEBOOK}-{ATTEMPT}.zip"
    files, hits = write_redacted_archive(ATTEMPT_DIR, path)
    print("evidence archive:", path, os.path.getsize(path), "bytes,", files, "files, redactions:", hits, flush=True)
    return path


_os = dict(l.split("=", 1) for l in (_read("/etc/os-release") or "").splitlines() if "=" in l)
_kaggle = {k: os.environ[k] for k in ("KAGGLE_KERNEL_RUN_TYPE", "KAGGLE_DOCKER_IMAGE") if k in os.environ}   # allow-list only; never the raw environment
mark("prelude", python=platform.python_version(), expect=EXPECT, gpu_device=HAS_GPU_DEVICE, os=_os.get("PRETTY_NAME", "").strip('"'), kaggle=_kaggle, inputs=_inputs)
print("ATTEMPT", ATTEMPT, "| markers:", ATTEMPT_DIR + "/markers.jsonl", flush=True)

In [ ]:
# GPU observation helpers: nvidia-smi in a SEPARATE process (this kernel never imports torch, pynvml or CUDA), plus /proc process-group states.
def gpu_query():
    q = "index,name,memory.used,memory.total,utilization.gpu,temperature.gpu,power.draw"
    try:
        r = subprocess.run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=30)
        return {"nvidia_smi_rc": r.returncode, "gpus": [dict(zip(q.split(","), (x.strip() for x in l.split(",")))) for l in r.stdout.strip().splitlines()]}
    except Exception as e:
        return {"nvidia_smi_error": type(e).__name__}


def group_procs(pgid):
    """Every process in process group `pgid`: pid, state, rss (MiB), command name. Read from /proc only."""
    page, out = os.sysconf("SC_PAGE_SIZE"), []
    for p in os.listdir("/proc"):
        if not p.isdigit():
            continue
        st = _read(f"/proc/{p}/stat")
        if not st:
            continue
        rest = st.rsplit(")", 1)[1].split()
        if int(rest[2]) == pgid:
            out.append({"pid": int(p), "state": rest[0], "rss_mib": round(int(rest[21]) * page / 2**20, 1), "comm": st[st.find("(") + 1:st.rfind(")")][:24]})
    return out


def inventory():
    import importlib.metadata as md
    norm = lambda n: re.sub(r"[-_.]+", "-", n).lower()                     # noqa: E731
    return {norm(d.metadata["Name"]): d.version for d in md.distributions() if d.metadata["Name"]}

In [ ]:
# C1. Application credentials (Kaggle Secrets) and the diagnostic transport to the OCI deployment. Shared by notebooks 01, 02 and 03 (generated from
# scripts/kaggle_cells/diag_credentials.py; CPU tests exercise this exact text). Standard library only: no cloud SDK, no cloud-provider credential.
#
# Two DIFFERENT application credentials, never interchangeable:
#   ENIGMA_DIAG_KEY       edx_<8>_<43>      56 chars  authenticates ONLY POST /api/v1/diag/events (notebooks 01, 02 oci-assisted, 03)
#   ENIGMA_RUNTIME_TOKEN  enig_rt_<8>_<43>  60 chars  authenticates ONLY Enigma runtime registration / heartbeats / telemetry / jobs (notebook 03)
# Neither is a cloud-provider key. Values are never printed, stored in os.environ, written to evidence, or sent anywhere but their own endpoint; only a
# category, a code and lengths are ever recorded. Only a VALIDATED value is cached; a failed lookup never poisons the cache. After fixing a secret in
# Kaggle's sidebar (Add-ons -> Secrets), call refresh_credentials() (or just re-run the preflight cell): the attempt id, the event sequence, completed
# verdicts and evidence files are all kept. Note: reading a Kaggle Secret is itself a request to Kaggle's own backend (kaggle_secrets talks to the
# Kaggle web service from inside the VM), so no secret is read during an offline observation window.
import re, json, time, socket, ssl, urllib.request, urllib.error, urllib.parse

DIAG_URL = ""
CRED_SPECS = {
    "ENIGMA_DIAG_KEY": {"category": "diagnostic_key", "pattern": r"edx_[a-z0-9]{8}_[A-Za-z0-9_-]{43}", "length": 56, "prefix": "edx_"},
    "ENIGMA_RUNTIME_TOKEN": {"category": "enigma_runtime_token", "pattern": r"enig_rt_[a-z0-9]{8}_[A-Za-z0-9_-]{43}", "length": 60, "prefix": "enig_rt_"},
}
CRED_GUIDANCE = {
    "SECRET_OK": "credential retrieved and validated (value not shown)",
    "SECRET_MISSING": "no Kaggle Secret with this exact name is available: add it in Add-ons -> Secrets, tick it for this notebook, then re-run this cell",
    "SECRET_ACCESS_DENIED": "the secret exists but this notebook may not read it: in Add-ons -> Secrets tick the checkbox next to it, then re-run this cell",
    "SECRET_EMPTY": "the Kaggle Secret is empty: paste the whole value again, then re-run this cell",
    "SECRET_FORMAT_INVALID": "the value does not have the expected format/length: paste exactly the one-line value (no quotes, no NAME=, no spaces), then re-run this cell",
    "SECRET_RETRIEVAL_FAILED": "Kaggle's secret service did not answer: check Internet is ON, then re-run this cell",
}
_WRONG_TYPE = {
    "ENIGMA_DIAG_KEY": ("Wrong credential type. ENIGMA_DIAG_KEY requires the diagnostic key from the local ENIGMA_DIAG_KEY.txt file, not the Enigma "
                        "runtime token."),
    "ENIGMA_RUNTIME_TOKEN": ("Wrong credential type. ENIGMA_RUNTIME_TOKEN requires the Enigma runtime key (enig_rt_...) issued in the Enigma web app "
                             "(Runtimes), not the diagnostic key."),
}
_AWS_ID = re.compile(r"(AKIA|ASIA)[0-9A-Z]{16}")
_CRED_CACHE = {}                                       # name -> validated value ONLY
_DIAG = {"seq": 0, "last": None}                      # the event sequence survives credential refreshes


def _secret_client():
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient()


def classify_secret(name, value):
    """(code, facts) for a raw secret value. facts never contain the value, a fragment of it, or its key id."""
    spec = CRED_SPECS[name]
    facts = {"credential": name, "expected_category": spec["category"], "expected_length": spec["length"]}
    if value is None:
        return "SECRET_MISSING", facts
    if not isinstance(value, str):
        return "SECRET_FORMAT_INVALID", facts
    v = value.strip()                                 # a trailing newline from copy/paste is tolerated; anything inside is not
    facts.update(actual_length=len(v), embedded_whitespace=any(c.isspace() for c in v), quoted=v[:1] in "'\"" and len(v) > 0,
                 looks_like_assignment=bool(re.match(r"^[A-Za-z_][A-Za-z0-9_]*\s*=", v)))
    if not v:
        return "SECRET_EMPTY", facts
    if re.fullmatch(spec["pattern"], v):
        facts["supplied_category"] = spec["category"]
        return "SECRET_OK", facts
    for other, ospec in CRED_SPECS.items():
        if other != name and (v.startswith(ospec["prefix"]) or re.search(ospec["pattern"], v)):
            facts["supplied_category"] = ospec["category"]
            return "SECRET_WRONG_TYPE", facts
    if _AWS_ID.search(v):
        facts["supplied_category"] = "aws_access_key"
        return "SECRET_WRONG_TYPE", facts
    facts["supplied_category"] = "unknown"
    return "SECRET_FORMAT_INVALID", facts


def _retrieval_code(exc):
    """Classify a kaggle_secrets exception from its class and message WITHOUT recording the message."""
    if isinstance(exc, ImportError):
        return "SECRET_RETRIEVAL_FAILED"
    text = (type(exc).__name__ + " " + str(exc)).lower()
    if "no user secrets exist" in text or "not found" in text or "does not exist" in text:
        return "SECRET_MISSING"
    if any(w in text for w in ("permission", "denied", "unauthori", "forbidden", "not attached", "not authorized", "403", "401")):
        return "SECRET_ACCESS_DENIED"
    return "SECRET_RETRIEVAL_FAILED"


def credential_message(code, name):
    return _WRONG_TYPE[name] if code == "SECRET_WRONG_TYPE" else CRED_GUIDANCE.get(code, code)


def get_credential(name, refresh=False):
    """(value or None, result). Caches only a validated value; any failure clears the cache entry so a later call retries."""
    if refresh:
        _CRED_CACHE.pop(name, None)
    if name in _CRED_CACHE:
        return _CRED_CACHE[name], {"credential": name, "code": "SECRET_OK", "cached": True, "message": CRED_GUIDANCE["SECRET_OK"]}
    try:
        raw = _secret_client().get_secret(name)
    except Exception as exc:                          # noqa: BLE001 - classified; the exception text is never recorded
        _CRED_CACHE.pop(name, None)
        code = _retrieval_code(exc)
        return None, {"credential": name, "code": code, "error_type": type(exc).__name__, "message": credential_message(code, name)}
    code, facts = classify_secret(name, raw)
    if code != "SECRET_OK":
        _CRED_CACHE.pop(name, None)
        return None, {**facts, "code": code, "message": credential_message(code, name)}
    _CRED_CACHE[name] = raw.strip()
    return _CRED_CACHE[name], {**facts, "code": code, "cached": False, "message": CRED_GUIDANCE["SECRET_OK"]}


def refresh_credentials(*names):
    """Forget cached credentials (all, or the named ones) so the next use re-reads Kaggle Secrets. Nothing else is reset."""
    for n in names or list(_CRED_CACHE):
        _CRED_CACHE.pop(n, None)
    return sorted(_CRED_CACHE)


def check_credential(name, step):
    """Preflight verdict for one credential: PASS only for a correctly typed, correctly formatted value."""
    _, res = get_credential(name, refresh=True)
    ev = {k: v for k, v in res.items() if k != "message"}
    if res["code"] == "SECRET_OK":
        verdict(step, "PASS", f"{name}: {res['message']}", **ev)
        return res
    print(f"CREDENTIAL {name}: {res['code']} -> {res['message']}", flush=True)
    verdict(step, "INCONCLUSIVE", f"{name}: {res['code']}", **ev)


# ------------------------------------------------------------------------------------------------------------- transport to the OCI diagnostic route
DIAG_GUIDANCE = {
    "DIAGNOSTIC_PERSISTENCE_VERIFIED": "the OCI deployment acknowledged a durable write (stored or already stored)",
    "DIAGNOSTIC_AUTH_REJECTED": "the diagnostic route rejected ENIGMA_DIAG_KEY: re-check the Kaggle Secret against the key issued by scripts/diag_keys.py; do not retry blindly",
    "DIAGNOSTIC_KEY_EXPIRED_OR_REVOKED": "ENIGMA_DIAG_KEY is correct but expired or revoked: ask for a newly issued diagnostic key",
    "DIAGNOSTIC_QUOTA_EXCEEDED": "this diagnostic key used its whole event quota: ask for a newly issued diagnostic key",
    "DIAGNOSTIC_NETWORK_TIMEOUT": "the diagnostic endpoint did not answer in time",
    "DIAGNOSTIC_DNS_ERROR": "the diagnostic endpoint's host name did not resolve (Internet OFF?)",
    "DIAGNOSTIC_TLS_ERROR": "TLS / certificate verification failed (verification is never disabled)",
    "DIAGNOSTIC_CONNECTION_ERROR": "the connection was refused or reset",
    "DIAGNOSTIC_HTTP_ERROR": "the diagnostic endpoint answered with an HTTP error",
    "DIAGNOSTIC_INVALID_RESPONSE": "the diagnostic endpoint answered with a malformed acknowledgement",
    "DIAGNOSTIC_PERSISTENCE_NOT_ACKNOWLEDGED": "the answer did not acknowledge every event as durably stored",
    "DIAGNOSTIC_CREDENTIAL_UNAVAILABLE": "no valid ENIGMA_DIAG_KEY: nothing was sent",
    "REMOTE_DISABLED": "remote diagnostics are disabled in this mode: nothing was sent",
}
_SERVER_ERRORS = {"unauthorized", "key_expired_or_revoked", "quota_exhausted", "too_large", "bad_json", "bad_events", "event_not_object",
                  "unknown_field", "missing_field", "bad_field", "bad_identity", "method_not_allowed", "unavailable"}
_TRANSIENT_HTTP = {429, 500, 502, 503, 504}
_TRANSIENT_CODES = {"DIAGNOSTIC_NETWORK_TIMEOUT", "DIAGNOSTIC_CONNECTION_ERROR"}
REMOTE_ENABLED = True                                  # notebook 02 standalone mode turns this off: then no secret is read and nothing is sent
_REMOTE_FIELDS = {"stage", "phase", "http", "latency_s", "target", "exc", "exit_code", "signal", "last_ok", "verdict", "note", "gpu_count",
                  "gpu_mem_used_mib", "gpu_util_pct", "wheel_build", "wheel_sha256", "artifact", "offset_s", "child_pid", "child_exit", "outcome"}


class DiagResult(dict):
    """Truthy only when the OCI deployment durably acknowledged every event."""
    def __bool__(self):
        return self.get("code") == "DIAGNOSTIC_PERSISTENCE_VERIFIED"


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    """Never follow a redirect: the Authorization header must never be replayed to another URL or host."""
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


def _build_opener():
    return urllib.request.build_opener(_NoRedirect(), urllib.request.HTTPSHandler(context=ssl.create_default_context()))


_OPENER = _build_opener()


def _diag_url_ok(url):
    u = urllib.parse.urlsplit(url)
    d = urllib.parse.urlsplit(DIAG_URL)
    # Exactly the generator-configured OCI host, https, the one diagnostic path; never another host (the Authorization header goes nowhere else).
    return bool(DIAG_URL) and u.scheme == "https" and bool(u.hostname) and u.hostname == d.hostname and u.port == d.port \
        and u.path == "/api/v1/diag/events" and not (u.username or u.password or u.query or u.fragment)


def _net_code(exc):
    r = exc.reason if isinstance(exc, urllib.error.URLError) and not isinstance(exc, urllib.error.HTTPError) else exc
    if isinstance(r, socket.gaierror):
        return "DIAGNOSTIC_DNS_ERROR", "dns"
    if isinstance(r, ssl.SSLError) or isinstance(r, ssl.CertificateError):
        return "DIAGNOSTIC_TLS_ERROR", "tls"
    if isinstance(r, (TimeoutError, socket.timeout)):
        return "DIAGNOSTIC_NETWORK_TIMEOUT", "connect" if r is not exc else "read"
    if isinstance(r, (ConnectionError, OSError)):
        return "DIAGNOSTIC_CONNECTION_ERROR", "connect"
    return "DIAGNOSTIC_CONNECTION_ERROR", "unknown"


def post_events(events, key, url=None, timeout=15.0, max_attempts=3, backoff=1.0):   # bounded; the server answers only after the insert committed
    """POST {"events": [...]} once per attempt. Retries ONLY transient failures, with the SAME body (same attempt id and seq): the server's
    conditional write makes a resubmission a duplicate, never a second record. Never retries an authentication or validation answer."""
    url = url or DIAG_URL
    body = json.dumps({"events": events}).encode()
    res = DiagResult(code="DIAGNOSTIC_HTTP_ERROR", attempts=0)
    for i in range(max_attempts):
        res = DiagResult(_post_once(url, body, key, timeout, len(events)), attempts=i + 1)
        if not res.get("transient") or i + 1 == max_attempts:
            return res
        time.sleep(backoff * (2 ** i))
    return res


def _post_once(url, body, key, timeout, n):
    req = urllib.request.Request(url, data=body, method="POST", headers={"Content-Type": "application/json", "Authorization": f"Bearer {key}",
                                                                          "User-Agent": "Enigma-Kaggle-Diagnostic/5.0"})
    t = time.monotonic()
    try:
        with _OPENER.open(req, timeout=timeout) as r:
            status, raw = r.status, r.read(4097)
    except urllib.error.HTTPError as e:
        try:
            raw = e.read(513)
        except Exception:                             # noqa: BLE001
            raw = b""
        finally:
            e.close()
        err = None
        try:
            j = json.loads(raw[:512])
            err = j.get("error") if isinstance(j, dict) and j.get("error") in _SERVER_ERRORS else None
        except ValueError:
            pass
        out = {"http": e.code, "server_error": err, "latency_s": round(time.monotonic() - t, 3)}
        if 300 <= e.code < 400:
            return {**out, "code": "DIAGNOSTIC_HTTP_ERROR", "detail": "redirect_refused"}
        if e.code == 401:
            return {**out, "code": "DIAGNOSTIC_KEY_EXPIRED_OR_REVOKED" if err == "key_expired_or_revoked" else "DIAGNOSTIC_AUTH_REJECTED"}
        if e.code == 429 and err == "quota_exhausted":
            return {**out, "code": "DIAGNOSTIC_QUOTA_EXCEEDED"}
        return {**out, "code": "DIAGNOSTIC_HTTP_ERROR", "transient": e.code in _TRANSIENT_HTTP}
    except (urllib.error.URLError, OSError) as e:      # DNS, TLS, refused, connect timeout (URLError) / read timeout, reset (OSError)
        code, phase = _net_code(e)
        return {"code": code, "phase": phase, "error_type": type(e.reason if isinstance(e, urllib.error.URLError) else e).__name__,
                "transient": code in _TRANSIENT_CODES, "latency_s": round(time.monotonic() - t, 3)}
    out = {"http": status, "latency_s": round(time.monotonic() - t, 3)}
    if status != 200:
        return {**out, "code": "DIAGNOSTIC_HTTP_ERROR"}
    if len(raw) > 4096:
        return {**out, "code": "DIAGNOSTIC_INVALID_RESPONSE", "detail": "oversized"}
    try:
        ack = json.loads(raw)
    except ValueError:
        return {**out, "code": "DIAGNOSTIC_INVALID_RESPONSE", "detail": "not_json"}
    ints = isinstance(ack, dict) and all(isinstance(ack.get(k), int) and not isinstance(ack.get(k), bool) and ack[k] >= 0 for k in ("stored", "duplicates"))
    if not ints:
        return {**out, "code": "DIAGNOSTIC_PERSISTENCE_NOT_ACKNOWLEDGED", "detail": "missing_or_invalid_fields"}
    out.update(stored=ack["stored"], duplicates=ack["duplicates"])
    if ack["stored"] + ack["duplicates"] != n:
        return {**out, "code": "DIAGNOSTIC_PERSISTENCE_NOT_ACKNOWLEDGED", "detail": "count_mismatch"}
    return {**out, "code": "DIAGNOSTIC_PERSISTENCE_VERIFIED"}


def _event(name, rec, fields, seq):
    ev = {"attempt": ATTEMPT, "nb": NOTEBOOK, "seq": seq, "marker": name, "utc": rec["utc"], "t": rec["t"], "experiment": EXPERIMENT,
          "pid": rec["pid"], "pgid": rec["pgid"], "sid": rec["sid"], "mem_available_mib": rec.get("memavailable_mib"),
          "cgroup_current_mib": rec.get("cgroup_current_mib"), "cgroup_max_mib": rec.get("cgroup_max_mib"),
          "oom_kill": rec.get("cgroup_events", {}).get("oom_kill"), "vm_uptime_s": rec.get("vm_uptime_s"),
          "oom_state": "available" if "oom_kill" in rec.get("cgroup_events", {}) else "unavailable", "transport": globals().get("RUNTIME_TRANSPORT")}
    ev.update({k: v for k, v in fields.items() if k in _REMOTE_FIELDS and v is not None})
    return {k: v for k, v in ev.items() if v is not None}


def rmark(name, **fields):
    """mark() locally, then (if remote diagnostics are enabled) one bounded event to the OCI diagnostic route. Returns a DiagResult: truthy only after it
    acknowledged the durable write. A 401 forgets the cached key (fix the secret, then re-run). Nothing secret is ever recorded."""
    rec = mark(name, **fields)
    if not REMOTE_ENABLED:
        return DiagResult(code="REMOTE_DISABLED", marker=name)
    key, cred = get_credential("ENIGMA_DIAG_KEY")
    if key is None:
        mark("remote_skipped", marker_name=name, credential_code=cred["code"])
        print(f"REMOTE NOT-SENT {name}: {cred['code']} -> {cred['message']}", flush=True)
        return DiagResult(code="DIAGNOSTIC_CREDENTIAL_UNAVAILABLE", credential_code=cred["code"], marker=name)
    _DIAG["seq"] += 1
    ev = _event(name, rec, fields, _DIAG["seq"])
    if not _diag_url_ok(DIAG_URL):
        return DiagResult(code="DIAGNOSTIC_HTTP_ERROR", detail="diag_url_not_allowed", marker=name, seq=ev["seq"])
    res = post_events([ev], key)
    res.update(marker=name, seq=ev["seq"])
    if res["code"] in ("DIAGNOSTIC_AUTH_REJECTED", "DIAGNOSTIC_KEY_EXPIRED_OR_REVOKED"):
        _CRED_CACHE.pop("ENIGMA_DIAG_KEY", None)
    _DIAG["last"] = {"event": ev, "result": dict(res)}
    mark("remote_result", **{f"remote_{k}": v for k, v in res.items()})
    print(f"REMOTE {'ACK' if res else 'NOT-ACKNOWLEDGED'} {name} seq={ev['seq']} {res['code']}"
          + ("" if res else f" -> {DIAG_GUIDANCE.get(res['code'], '')}"), flush=True)
    return res


def resubmit_last():
    """Resend the last event UNCHANGED (same attempt id, same seq). A durable record answers duplicates == 1."""
    last = _DIAG.get("last")
    key, _ = get_credential("ENIGMA_DIAG_KEY")
    if not last or key is None:
        return DiagResult(code="DIAGNOSTIC_CREDENTIAL_UNAVAILABLE" if last else "DIAGNOSTIC_PERSISTENCE_NOT_ACKNOWLEDGED")
    res = post_events([last["event"]], key)
    res.update(marker=last["event"]["marker"], seq=last["event"]["seq"])
    mark("remote_resubmit_result", **{f"remote_{k}": v for k, v in res.items()})
    return res


def rcheck(step, name, **fields):
    """A marker that MUST be durable before the next (risky or expensive) operation. Local-only when remote diagnostics are disabled."""
    res = rmark(name, **fields)
    if not REMOTE_ENABLED:
        return res
    if not res:
        verdict(step, "INCONCLUSIVE", f"{name}: {res['code']} -> {DIAG_GUIDANCE.get(res['code'], '')}", remote_code=res["code"])
    return res

In [ ]:
# Runtime-endpoint capability probe (shared; generated from scripts/kaggle_cells/capability.py). ONE bounded, unauthenticated GET of the SELECTED
# endpoint's public /api/v1/health: https only, an explicit host allow-list, TLS verification, no redirect, no credential header, a bounded body and a
# bounded WALL-CLOCK time. Four results are reported independently and an HTTP 200 alone is never readiness:
#   CAPABILITY_HTTP_RESULT              the request itself (status / transport error class)
#   CAPABILITY_SCHEMA_RESULT            the body is the expected JSON object with typed fields and the expected API contract
#   CAPABILITY_DEPLOYMENT_RESULT        deployment label, store, auth provider AND the data-plane fingerprint equal the values expected for the mode
#   CAPABILITY_DURABLE_EVIDENCE_RESULT  the OCI deployment acknowledged BOTH the before- and the after-probe diagnostic markers
import ssl, threading, urllib.error, urllib.parse, urllib.request

HEALTH_SCHEMA = {"ok": bool, "version": str, "api_contract": int, "store": str, "auth": str, "deployment": str, "data_plane": str}


def _cap_opener():
    class _NoRedirectCap(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            return None
    return urllib.request.build_opener(_NoRedirectCap(), urllib.request.HTTPSHandler(context=ssl.create_default_context()))


def probe_health(base_url, allowed_host, wall_timeout=20.0, max_bytes=4096, opener=None):
    """Returns {"http": {...}, "body": dict|None, "latency_s": float}. Never raises; never sends a credential."""
    u = urllib.parse.urlsplit(base_url)
    if u.scheme != "https" or u.hostname != allowed_host or u.username or u.password or u.port not in (None, 443):
        return {"http": {"result": "FAIL", "error": "endpoint_not_allowed"}, "body": None, "latency_s": 0.0}
    req = urllib.request.Request(base_url.rstrip("/") + "/api/v1/health", method="GET",
                                 headers={"User-Agent": "Enigma-Kaggle-Capability/7.0", "Accept": "application/json"})
    assert not any(k.lower() in ("authorization", "cookie") for k, _ in req.header_items()), "a public health probe never carries a credential"
    box, t0 = {}, time.monotonic()

    def run():
        try:
            with (opener or _cap_opener()).open(req, timeout=max(1.0, wall_timeout - 1.0)) as r:
                box["status"], box["raw"] = r.status, r.read(max_bytes + 1)
        except urllib.error.HTTPError as e:
            box["status"] = e.code                        # 3xx lands here too: redirects are never followed
        except (ssl.SSLError, ssl.CertificateError) as e:
            box["error"] = "tls:" + type(e).__name__
        except Exception as e:
            box["error"] = type(e).__name__
    th = threading.Thread(target=run, daemon=True)
    th.start()
    th.join(wall_timeout)
    lat = round(time.monotonic() - t0, 3)
    if th.is_alive():
        return {"http": {"result": "FAIL", "error": "wall_clock_timeout"}, "body": None, "latency_s": lat}
    if "error" in box:
        return {"http": {"result": "FAIL", "error": box["error"]}, "body": None, "latency_s": lat}
    st = box.get("status")
    if st != 200:
        return {"http": {"result": "FAIL", "status": st, "error": "redirect_refused" if st and 300 <= st < 400 else "http_status"}, "body": None, "latency_s": lat}
    raw = box.get("raw", b"")
    if len(raw) > max_bytes:
        return {"http": {"result": "FAIL", "status": st, "error": "body_too_large"}, "body": None, "latency_s": lat}
    try:
        body = json.loads(raw)
    except ValueError:
        body = None
    return {"http": {"result": "PASS", "status": st}, "body": body, "latency_s": lat}


def check_schema(body, min_contract):
    if not isinstance(body, dict):
        return {"result": "FAIL", "why": "body is not a JSON object"}
    bad = [k for k, t in HEALTH_SCHEMA.items() if not isinstance(body.get(k), t) or (t is int and isinstance(body.get(k), bool))]
    if bad:
        return {"result": "FAIL", "why": f"missing or mistyped fields {bad}"}
    if body["ok"] is not True:
        return {"result": "FAIL", "why": "ok is not true"}
    if body["api_contract"] < min_contract:
        return {"result": "FAIL", "why": f"api_contract {body['api_contract']} < required {min_contract}"}
    return {"result": "PASS", "why": f"api_contract {body['api_contract']}, version {body['version']}"}


def check_deployment(body, expect):
    """expect = {"deployment", "store", "auth", "data_plane"}: every value must match exactly (the data-plane fingerprint is fixed at notebook
    generation time from the deployment's configured store, so a staging endpoint can never pass as production and vice versa)."""
    if not isinstance(body, dict):
        return {"result": "FAIL", "why": "no body"}
    diff = {k: [body.get(k), v] for k, v in expect.items() if body.get(k) != v}
    return {"result": "PASS" if not diff else "FAIL", "why": "all expected" if not diff else f"got/expected {diff}"}

## 03-0. GPU environment guard (local only)

In [ ]:
# 03-0. GPU environment guard (local only: nvidia-smi in a separate process, /proc, a loopback socket; no network, no CUDA in this kernel).
# The same inventory approach as the verified notebook 02, plus SM75, free GPU memory, host memory and "no inference server already running".
EXPERIMENT = f"build5-{INTEGRATION_MODE}"              # every later MARK names the integration mode
assert INTEGRATION_MODE in ("oci-production",), INTEGRATION_MODE
import importlib.metadata as md, socket
g = gpu_query()
names = [x.get("name") for x in g.get("gpus", [])]
try:
    _r = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap,memory.free", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=30)
    caps = [tuple(x.strip() for x in l.split(",")) for l in _r.stdout.strip().splitlines()]
except Exception as e:
    caps = [("error:" + type(e).__name__, "0")]
vers = {p: (md.version(p) if p in {d.metadata["Name"].lower() for d in md.distributions()} else None) for p in ("torch", "triton", "transformers")}
EXPECTED_VERSIONS = {"torch": "2.11.0+cu130", "triton": "3.6.0", "transformers": "5.20.2"}
_h = host()
with socket.socket() as _s:
    _s.settimeout(1.0)
    port_busy = _s.connect_ex(("127.0.0.1", 8000)) == 0
servers = []
for _p in os.listdir("/proc"):
    if _p.isdigit() and int(_p) != os.getpid():
        _c = (_read(f"/proc/{_p}/cmdline") or "").replace("\x00", " ")
        if "vllm" in _c and ("serve" in _c or "entrypoints" in _c):
            servers.append(int(_p))
checks = {"python_3_13": sys.version_info[:2] == (3, 13), "two_tesla_t4": names == ["Tesla T4", "Tesla T4"],
          "sm75": [c[0] for c in caps] == ["7.5", "7.5"], "gpu_free_mib_ge_14000": all(c[1].isdigit() and int(c[1]) >= 14000 for c in caps) and len(caps) == 2,
          "host_available_mib_ge_24000": (_h.get("memavailable_mib") or 0) >= 24000, "versions_as_validated": vers == EXPECTED_VERSIONS,
          "no_local_server": not port_busy and not servers}
mark("environment", integration_mode=INTEGRATION_MODE, python=platform.python_version(), gpus=names, compute_caps=caps, versions=vers, port_8000_busy=port_busy,
     vllm_pids=servers, checks=checks, **g)
bad = [k for k, v in checks.items() if not v]
verdict("environment", "PASS" if not bad else "INCONCLUSIVE",
        f"{names}, SM {[c[0] for c in caps]}, free MiB {[c[1] for c in caps]}, {vers}" + (f"; NOT as validated: {bad} (start a NEW GPU T4 x2 session; "
        "a version difference means the historical baseline does not cover this image)" if bad else "; matches the validated notebook 02 environment"),
        checks=checks)

## 03-1. Credential preflight and a durable OCI diagnostic acknowledgement (each credential cell re-runnable on its own)

In [ ]:
require("environment")
refresh_credentials("ENIGMA_DIAG_KEY")
check_credential("ENIGMA_DIAG_KEY", "diag_key_preflight")

In [ ]:
refresh_credentials("ENIGMA_RUNTIME_TOKEN")
check_credential("ENIGMA_RUNTIME_TOKEN", "runtime_token_preflight")   # format/type only; the live check is 03-4

In [ ]:
require("diag_key_preflight")
if not DIAG_URL:
    verdict("diag_endpoint", "INCONCLUSIVE", "BLOCKED: the OCI endpoint is not deployed yet (OCI_BASE_URL is empty in the generator); nothing was sent",
            blocked=True)
first = rmark("diag_first_event", stage="diag", phase="probe", note="first diagnostic event")
if not first:
    verdict("diag_endpoint", "INCONCLUSIVE", f"{first['code']}: {DIAG_GUIDANCE.get(first['code'], '')}", remote=dict(first))
dup = resubmit_last()                   # the SAME event (same attempt id and seq) again: a durable record must answer as an already-stored duplicate
durable = bool(dup) and dup.get("duplicates") == 1 and dup.get("stored") == 0
verdict("diag_endpoint", "PASS" if durable else "INCONCLUSIVE",
        f"first {first['code']} (stored={first.get('stored')}, duplicates={first.get('duplicates')}); identical resubmission {dup.get('code')} "
        f"(duplicates={dup.get('duplicates')})", first=dict(first), resubmit=dict(dup))

## 03-2. Historical baseline evidence (manifest from the executed notebooks 01 and 02)

In [ ]:
# 03-2. HISTORICAL baseline evidence (not an operator checkbox): a small manifest derived by scripts/build_baseline_manifest.py from the owner's
# executed notebook 01 / 02 runs and their diagnostic archives (hashes of the source files are inside). Its digest is re-verified here. This proves what
# PASSED in those earlier sessions only; this session still verifies its own environment, the wheel bytes, the install and real local inference.
require("environment")
import hashlib
BASELINE = json.loads('{"acceptance":{"BUILD5_DEPENDENCY_CONFLICTS_INTRODUCED":0,"BUILD5_LOCAL_GPU_INFERENCE":"PASS","BUILD5_SAFE_SHUTDOWN":"PASS","BUILD5_SERVER_600S_STABILITY":"PASS","BUILD5_WHEEL_SHA256":"VERIFIED","NOTEBOOK_01_AWS_ONLY_BASELINE":"PASS"},"generated_utc":"2026-11-27T01:24:44Z","kind":"historical baseline acceptance (NOT current-session readiness)","manifest_sha256":"51a8671dfd0dfa240a747ff0b0ef3b9403871f81cc20d0ab4182430504a3e388","nb01":{"attempt":"20261127T001031Z-29f49dac","aws_survival":"observed 600/600s, oom_kill delta 0, remote 10/10 acknowledged, problems []","executed_code_cells":14,"experiment":"no-sdk-expA","first_utc":"2026-11-27T00:10:31Z","gpu_device":false,"inputs":[],"last_utc":"2026-11-27T00:37:36Z","mode":"aws-required","oom_kill_max":0,"peak_cgroup_mib":1157.9,"python":"3.13.18","railway_requests":0,"records":247,"transport":"none","verdicts":{"aws_endpoint":"PASS","aws_survival":"PASS","controls_https":"PASS","controls_survival_120s":"PASS","diag_key_preflight":"PASS","idle_10min":"PASS","overall":"PASS"}},"nb02":{"attempt":"20261127T004108Z-4acaca8a","aws_requests":0,"executed_code_cells":13,"experiment":"build5-standalone","first_utc":"2026-11-27T00:41:08Z","gpus":["Tesla T4","Tesla T4"],"install":{"conflicts_after":8,"first_plan_stopped_for_acknowledgement":true,"installed_build":"5","introduced_conflicts":[],"preexisting_conflicts":8,"replaced":{"pydantic-settings":["2.16.3","2.16.2"]}},"last_utc":"2026-11-27T01:02:14Z","local_inference":"32 chunks, generated-token counter +33.0","mode":"standalone","model_config":{"attention":"TRITON_ATTN","dtype":"float16","enforce_eager":"True","gpu_memory_utilization":0.6,"gpu_memory_utilization_source":"notebook 02 server cell (StokastiskConfig gpu_memory_utilization=0.60)","max_seq_len":"2048","model":"Qwen/Qwen2.5-0.5B-Instruct","pipeline_parallel_size":"1","tensor_parallel_size":"1"},"observation":{"observed_s":600.1,"oom_kill_delta":0,"ticks":40},"oom_kill_max":0,"peak_cgroup_mib":8046.3,"python":"3.13.16","railway_requests":0,"records":161,"server_log_notes":{"leaked_semaphore_warning":true,"process_manager_force_killed_remaining":true,"runtime_triton_jit_warnings":2},"shutdown":{"children_survived_leader":false,"exit_code":0,"graceful":true,"group_gone":true,"signals":["SIGINT"]},"transport":"none","verdicts":{"environment":"PASS","install":"PASS","install_plan":"PASS","local_inference":"PASS","remote_ready":"PASS","server_passive_10min":"PASS","server_shutdown":"PASS","server_start":"PASS","wheel_verified":"PASS"},"versions":{"torch":"2.11.0+cu130","transformers":"5.20.2","triton":"3.6.0"},"wheel":{"file":"stokastisk-0.1.0-5-cp313-cp313-linux_x86_64.whl","name_version":["stokastisk","2.4"],"native_libraries":["vllm/_C_stable_libtorch.abi3.so","vllm/_moe_C_stable_libtorch.abi3.so","vllm/_rust_tool_parser.abi3.so","vllm/cumem_allocator.abi3.so","vllm/fs_io_C.abi3.so","vllm/spinloop.abi3.so"],"pins":["pydantic>=2.12.0","pydantic-settings==2.16.2"],"sha256":"fe80d7c36fa78cf8a26feb1e00bb5f1f006ca45a027450353603976320ae0b3e","size":38057515,"tag":"cp313-cp313-linux_x86_64","wheel_build_tag":"5"}},"nb03_previous":{"attempt":"20261127T010840Z-8d84df3d","executed_code_cells":8,"outcome":"INTERRUPTED after runtime_capabilities (Railway transport); build 5 not installed","verdicts":{"aws_endpoint":"PASS","baselines_confirmed":"PASS","diag_key_preflight":"PASS","runtime_capabilities":"PASS","runtime_token_preflight":"PASS"}},"schema":1,"sources":{"nb01_archive":{"file":"01-enigma-kaggle-platform-aws-diagnostics-enigma-diag-attempts.zip","sha256":"9e21adee5e83a248bdd58e62286cc8dc6ca5f53e245641abcaeacef69d20e2f4","size":19916},"nb01_notebook":{"file":"01-enigma-kaggle-platform-aws-diagnostics.ipynb","sha256":"ca99a58c6f0a8bebb7c208237c84af0e13fc30499e2ec8eebe963d19890e5fd1","size":279531},"nb02_archive":{"file":"02-stokastisk-build5-kaggle-t4-validation-enigma-diag-attempts.zip","sha256":"fc24249ef5204f4b221b7c059cc44ed5b4ebe75810ceae2751dc1bb68d986358","size":37083330},"nb02_notebook":{"file":"02-stokastisk-build5-kaggle-t4-validation.ipynb","sha256":"8d4af7e70ccccf855b95fe36ba598b9b7d82141e8b74f6cdfe91736d93c73e76","size":219804},"nb03_previous_notebook":{"file":"03-enigma-stokastisk-build5-aws-integration.ipynb","sha256":"787762297e7b7be8063b7a5eef47861ab8f954c29560aead7a68ae3dad2da1cb","size":79395}}}')
_core = {k: v for k, v in BASELINE.items() if k not in ("generated_utc", "manifest_sha256")}
_digest = hashlib.sha256(json.dumps(_core, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
acc, b2 = BASELINE["acceptance"], BASELINE["nb02"]
checks = {"manifest_digest": _digest == BASELINE["manifest_sha256"] == "51a8671dfd0dfa240a747ff0b0ef3b9403871f81cc20d0ab4182430504a3e388",
          "nb01_expA_overall": acc["NOTEBOOK_01_AWS_ONLY_BASELINE"] == "PASS" and BASELINE["nb01"]["railway_requests"] == 0,
          "nb02_local_inference": acc["BUILD5_LOCAL_GPU_INFERENCE"] == "PASS", "nb02_600s_stability": acc["BUILD5_SERVER_600S_STABILITY"] == "PASS",
          "nb02_safe_shutdown": acc["BUILD5_SAFE_SHUTDOWN"] == "PASS", "no_conflict_introduced": acc["BUILD5_DEPENDENCY_CONFLICTS_INTRODUCED"] == 0,
          "wheel_identity": (b2["wheel"]["sha256"], b2["wheel"]["size"], b2["wheel"]["wheel_build_tag"]) == ("fe80d7c36fa78cf8a26feb1e00bb5f1f006ca45a027450353603976320ae0b3e", 38057515, "5"),
          "model_config": (b2["model_config"]["model"], b2["model_config"]["tensor_parallel_size"], b2["model_config"]["pipeline_parallel_size"],
                           b2["model_config"]["dtype"], b2["model_config"]["attention"], b2["model_config"]["max_seq_len"]) == ("Qwen/Qwen2.5-0.5B-Instruct", "1", "1", "float16", "TRITON_ATTN", "2048"),
          "environment_matches_baseline": b2["versions"] == vers and b2["gpus"] == names}
mark("baseline_manifest", digest=_digest, attempts=[BASELINE["nb01"]["attempt"], b2["attempt"]], acceptance=acc, checks=checks)
bad = [k for k, v in checks.items() if not v]
verdict("baseline_manifest", "PASS" if not bad else "INCONCLUSIVE",
        f"HISTORICAL: nb01 {BASELINE['nb01']['attempt']} expA PASS; nb02 {b2['attempt']} inference/600 s/shutdown PASS; wheel {b2['wheel']['sha256'][:16]}..."
        + (f"; FAILED checks {bad}" if bad else "; current environment equals the validated one"), checks=checks)

## 03-3. Capability gate (selected endpoint only) · 03-4. real runtime authentication · 03-5. post-capability survival

In [ ]:
# 03-3. Capability gate: EVIDENCE from the OCI deployment only. One public, unauthenticated health probe (no credential), bracketed by REQUIRED
# durable diagnostic markers. BLOCKED (no request at all) while OCI_BASE_URL is empty. HTTP 200 alone is never readiness: the schema, the deployment
# label, the store, the auth provider and the data-plane fingerprint fixed at generation time must all match.
require("baseline_manifest", "diag_endpoint", "runtime_token_preflight")
MODES = {"oci-production": {"url": "", "expect": {"deployment": "oci-a1", "store": "postgres", "auth": "betterauth", "data_plane": "1a21069207c5c12c"}}}
SCOPE = "PRODUCTION"
API_URL = MODES[INTEGRATION_MODE]["url"]                # fixed for the attempt; the connector is given only this URL and never falls back
if not API_URL:
    verdict("runtime_capabilities", "INCONCLUSIVE", "BLOCKED: the OCI deployment does not exist yet (OCI_BASE_URL is empty). No request was sent and no GPU "
            "work runs; notebook 02 standalone remains available. Do not bypass this gate.", blocked=True)
ALLOWED_HOST = urllib.parse.urlsplit(API_URL).hostname
rcheck("capability_marker", "before_capability_probe", stage="capability", phase="before", target=RUNTIME_TRANSPORT, note=INTEGRATION_MODE)
probe = probe_health(API_URL, ALLOWED_HOST, wall_timeout=20.0)
body = probe["body"] if isinstance(probe["body"], dict) else {}
sch = check_schema(probe["body"], 2) if probe["http"]["result"] == "PASS" else {"result": "FAIL", "why": "no HTTP 200"}
dep = check_deployment(body, MODES[INTEGRATION_MODE]["expect"]) if sch["result"] == "PASS" else {"result": "FAIL", "why": "schema not valid"}
after = rmark("after_capability_probe", stage="capability", phase="after", target=RUNTIME_TRANSPORT, http=probe["http"].get("status"),
              latency_s=probe["latency_s"], note=f"{INTEGRATION_MODE} http={probe['http']['result']} schema={sch['result']} deployment={dep['result']}")
CAP = {"CAPABILITY_HTTP_RESULT": probe["http"], "CAPABILITY_SCHEMA_RESULT": sch, "CAPABILITY_DEPLOYMENT_RESULT": dep,
       "CAPABILITY_DURABLE_EVIDENCE_RESULT": {"result": "PASS" if after else "FAIL", "before": "acknowledged", "after": after.get("code")}}
for k, v in CAP.items():
    print(f"{k}: {v['result']} | {json.dumps({x: y for x, y in v.items() if x != 'result'})}", flush=True)
ok = all(v["result"] == "PASS" for v in CAP.values())
verdict("runtime_capabilities", "PASS" if ok else "INCONCLUSIVE",
        f"{SCOPE} via {RUNTIME_TRANSPORT}: " + ", ".join(f"{k.split('_')[1]}={v['result']}" for k, v in CAP.items())
        + (f" | deployment={body.get('deployment')!r} store={body.get('store')!r} auth={body.get('auth')!r}" if body else "")
        + ("" if ok else " | NOT the expected data plane: no GPU or cloud stage runs"), capability={k: v["result"] for k, v in CAP.items()}, scope=SCOPE,
        deployment=body.get("deployment"), store=body.get("store"), auth=body.get("auth"), data_plane=body.get("data_plane"))

In [ ]:
# 03-4. Runtime credential: ACTUAL authentication against the selected endpoint, before any GPU work. One read-only, side-effect-free
# GET /api/v1/runtimes/me (no registration, no job operation). Requires a REAL (non-mock) runtime whose token was issued by the OCI deployment's web app
# (tokens issued by the retired Railway deployment are migrated with their hashes; a revoked or expired token fails here with 401).
require("runtime_capabilities")
rcheck("identity_marker", "before_runtime_identity", stage="identity", phase="before", target=RUNTIME_TRANSPORT)
_tok, _res = get_credential("ENIGMA_RUNTIME_TOKEN")
if _tok is None:
    verdict("runtime_identity", "INCONCLUSIVE", f"ENIGMA_RUNTIME_TOKEN {_res['code']}: {_res['message']}")
_req = urllib.request.Request(API_URL.rstrip("/") + "/api/v1/runtimes/me", headers={"Authorization": f"Bearer {_tok}", "Accept": "application/json",
                                                                                     "User-Agent": "Enigma-Kaggle-Capability/7.0"})
del _tok
who, st, t = {}, None, time.monotonic()
try:
    with _cap_opener().open(_req, timeout=20) as r:
        st, who = r.status, json.loads(r.read(4096))
except urllib.error.HTTPError as e:
    st = e.code
except Exception as e:
    st = "exc:" + type(e).__name__
del _req
who = who if isinstance(who, dict) else {}
after = rmark("after_runtime_identity", stage="identity", phase="after", http=st if isinstance(st, int) else None, latency_s=round(time.monotonic() - t, 3),
              note=f"is_mock={who.get('is_mock')}")
ok = st == 200 and who.get("is_mock") is False and bool(who.get("runtime_id")) and bool(after)
hint = {401: "the token is not accepted by THIS endpoint (wrong token for the mode, revoked, or expired)",
        404: "the endpoint does not serve /runtimes/me yet (API contract too old)"}.get(st, "")
verdict("runtime_identity", "PASS" if ok else "INCONCLUSIVE",
        f"http {st}; real runtime={who.get('is_mock') is False}; durable after-marker={bool(after)}" + (f" | {hint}" if hint and not ok else ""),
        runtime_id=who.get("runtime_id"), is_mock=who.get("is_mock"))

In [ ]:
# 03-5. Post-capability survival gate. Earlier fatal Kaggle sessions ended seconds after a cloud response (cause NOT established); nothing is installed until this
# session demonstrably survives its endpoint contact. Durable start marker -> 90 s PASSIVE window (local, fsynced, printed ticks every 2 s for 30 s,
# then every 5 s; no network) -> durable completion marker. If the whole VM is terminated, only the markers already stored on the OCI deployment remain.
require("runtime_identity")
observe("post_capability_survival", 90, every=5, dense_first_s=30, dense_every=2, durable_begin=True)
rcheck("post_capability_survival_marker", "post_capability_survival_complete", stage="survival", phase="after", verdict="PASS")

## 03-6. Immutable build 5 (exactly the notebook 02 workflow), server, real local inference

In [ ]:
require("post_capability_survival")
rcheck("remote_install_begin", "install_begin", stage="install", phase="before")
# Immutable Stokastisk build 5 ONLY. The wheel is downloaded and its exact SIZE and SHA-256 are verified BEFORE pip sees it (the version string 0.1.0
# does not distinguish build tags). Dry-run plan first; the one known replacement must be explicitly acknowledged; then install from the verified file,
# a before/after inventory diff, pip check, and the INSTALLED identity (WHEEL build tag) read without importing anything.
WHEEL_URL = "https://huggingface.co/waqasm86/stokastisk/resolve/main/wheels/stokastisk-0.1.0-5-cp313-cp313-linux_x86_64.whl"
WHEEL_SHA256 = "fe80d7c36fa78cf8a26feb1e00bb5f1f006ca45a027450353603976320ae0b3e"
WHEEL_SIZE = 38057515
KNOWN_REPLACEMENTS = {"pydantic-settings": ["2.16.3", "2.16.2"]}
ACKNOWLEDGE_PYDANTIC_SETTINGS_DOWNGRADE = False      # set True deliberately: build 5 pins pydantic-settings==2.16.2; Kaggle ships 2.16.3 (2.16.3 is a security release)
import hashlib, urllib.request
norm = lambda n: re.sub(r"[-_.]+", "-", n).lower()                                     # noqa: E731
BASE = ("torch", "triton", "transformers", "pydantic", "pydantic-core", "fastapi", "starlette", "numpy")
wheel_path = f"{ATTEMPT_DIR}/stokastisk-0.1.0-5-cp313-cp313-linux_x86_64.whl"
mark("wheel_download_begin", url=WHEEL_URL)
h, n = hashlib.sha256(), 0
with urllib.request.urlopen(WHEEL_URL, timeout=120) as r, open(wheel_path, "wb") as out:
    while True:
        chunk = r.read(1 << 20)
        if not chunk:
            break
        h.update(chunk); n += len(chunk); out.write(chunk)
got = h.hexdigest()
mark("wheel_downloaded", size=n, sha256=got)
if n != WHEEL_SIZE or got != WHEEL_SHA256:
    verdict("wheel_verified", "FAIL", f"NOT the published build 5: size {n} sha256 {got}")
verdict("wheel_verified", "PASS", f"published build 5: {n} bytes, sha256 {got[:16]}...", wheel_sha256=got)


def dependents(name):
    import importlib.metadata as md
    out = []
    for d in md.distributions():
        for req in d.requires or []:
            if norm(re.split(r"[ ;<>=!~\[(]", req.strip(), maxsplit=1)[0]) == name:
                out.append(f"{d.metadata['Name']}=={d.version}: {req}")
    return out


def pip_check():
    r = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
    return r.returncode, sorted({l.strip() for l in r.stdout.splitlines() if l.strip() and "no broken requirements" not in l.lower()})


pre_rc, pre_conflicts = pip_check()                    # Kaggle's OWN pre-existing conflicts are recorded, not blamed on build 5
mark("pip_check_before", rc=pre_rc, conflicts=pre_conflicts[:50])
before = inventory()
json.dump(before, open(f"{ATTEMPT_DIR}/packages-before.json", "w"), indent=1, sort_keys=True)
report = f"{ATTEMPT_DIR}/pip-dry-run-report.json"
r = subprocess.run([sys.executable, "-m", "pip", "install", "--dry-run", "--quiet", "--report", report, "--index-url", "https://pypi.org/simple", wheel_path],
                   capture_output=True, text=True)
if r.returncode != 0:
    verdict("install_plan", "FAIL", f"pip dry-run failed rc={r.returncode}", stderr_tail=r.stderr[-1500:])
plan = {norm(i["metadata"]["name"]): i["metadata"]["version"] for i in json.load(open(report))["install"]}
replaced = {k: [before[k], v] for k, v in plan.items() if k in before and before[k] != v}
mark("install_plan", would_install=len(plan), replaced=replaced, dependents={k: dependents(k) for k in replaced})
print("DEPENDENCY_VERSION_PRESERVATION:", "VERIFIED" if not replaced else f"CONFLICT {replaced}", flush=True)
unexpected = {k: v for k, v in replaced.items() if KNOWN_REPLACEMENTS.get(k) != v}
if unexpected:
    verdict("install_plan", "INCONCLUSIVE", f"build 5 would replace unexpected preinstalled packages {unexpected}: not installing", replaced=replaced)
if replaced and not ACKNOWLEDGE_PYDANTIC_SETTINGS_DOWNGRADE:
    verdict("install_plan", "INCONCLUSIVE", f"CONFLICT {replaced}: set ACKNOWLEDGE_PYDANTIC_SETTINGS_DOWNGRADE = True to accept it deliberately", replaced=replaced)
verdict("install_plan", "PASS", f"{len(plan)} to install; replacements {replaced or 'none'}", replaced=replaced)

mark("install_begin")
r = subprocess.run([sys.executable, "-m", "pip", "install", "--index-url", "https://pypi.org/simple", wheel_path], capture_output=True, text=True)
open(f"{ATTEMPT_DIR}/pip-install.log", "w").write(r.stdout[-200000:] + "\n--- stderr ---\n" + r.stderr[-50000:])
after = inventory()
changed = {k: [before[k], after[k]] for k in before if k in after and before[k] != after[k]}
removed = sorted(k for k in before if k not in after)
post_rc, post_conflicts = pip_check()
new_conflicts = sorted(set(post_conflicts) - set(pre_conflicts))
json.dump({"before": pre_conflicts, "after": post_conflicts, "introduced": new_conflicts}, open(f"{ATTEMPT_DIR}/pip-check.json", "w"), indent=1)
import importlib.metadata as md
_dist = md.distribution("stokastisk")
_build = next((l.split(":", 1)[1].strip() for l in (_dist.read_text("WHEEL") or "").splitlines() if l.startswith("Build:")), None)
mark("installed_identity", wheel_build=_build, version=_dist.version, changed=changed, removed=removed, pip_check_rc_before=pre_rc,
     pip_check_rc_after=post_rc, introduced_conflicts=new_conflicts)
base_same = all(before.get(b) == after.get(b) for b in BASE)
ok = (r.returncode == 0 and base_same and changed == replaced and not removed and _build == "5" and _dist.version == "0.1.0"
      and not new_conflicts)                           # PASS never hides a dependency incompatibility that the install introduced
verdict("install", "PASS" if ok else "FAIL", f"rc={r.returncode}, base unchanged={base_same}, changed={changed}, installed build={_build}, "
        f"new pip-check conflicts={new_conflicts or 'none'} (pre-existing: {len(pre_conflicts)})", installed_build=_build, changed=changed,
        introduced_conflicts=new_conflicts)

rcheck("remote_install_end", "install_end", stage="install", phase="after", wheel_build=_build, wheel_sha256=WHEEL_SHA256)

In [ ]:
require("install")
rcheck("remote_server_begin", "server_start_begin_remote", stage="server", phase="before")
# Start the Stokastisk server with the previously successful settings (TP=1, PP=1, 2048 tokens, gpu_memory_utilization 0.60, Triton attention, fp16, eager).
require("install")
for _k in ("no_proxy", "NO_PROXY"):                 # build 5: loopback HTTP honours http_proxy unless no_proxy lists it; set BEFORE the first request
    _have = [x for x in os.environ.get(_k, "").split(",") if x.strip()]
    os.environ[_k] = ",".join(_have + [h for h in ("127.0.0.1", "localhost", "::1") if h not in _have])
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
from stokastisk import StokastiskConfig, serving
from stokastisk.serving import ServerProcess
cfg = StokastiskConfig(tensor_parallel_size=1, pipeline_parallel_size=1, max_model_len=2048, gpu_memory_utilization=0.60)
assert (cfg.attention_backend, cfg.dtype, cfg.enforce_eager) == ("TRITON_ATTN", "half", True), "unexpected server defaults"
assert globals().get("server") is None, "a server already exists in this kernel: start a NEW session"
server = ServerProcess(MODEL, cfg, host="127.0.0.1", port=8000, served_model_name=MODEL, log_path=f"{ATTEMPT_DIR}/vllm-server.log")
mark("server_start_begin")
try:
    server.start_and_wait(timeout=900)
except BaseException as e:
    mark("server_start_failed", error=type(e).__name__, log_tail=server.log_tail(20))
    verdict("server_start", "FAIL", f"server did not become healthy: {type(e).__name__}")
procs = group_procs(server.pid)
mark("server_started", server_pid=server.pid, group=procs, **gpu_query())
verdict("server_start", "PASS" if server.health() else "FAIL", f"pid {server.pid}, {len(procs)} process(es) in its group", server_pid=server.pid)

rcheck("remote_server_started", "server_started_remote", stage="server", phase="after")

In [ ]:
# The previously verified local inference test: streamed reply and a rising native vLLM generated-token counter. No Enigma traffic.
require("server_start")
before_m = serving.fetch_metrics(server.base_url)
chunks = list(serving.stream_chat(server.base_url, MODEL, [{"role": "user", "content": "Count from one to ten in words."}], max_tokens=64))
time.sleep(2.5)
after_m = serving.fetch_metrics(server.base_url)
key = next((k for k in after_m if k.startswith("vllm:generation_tokens")), None)
delta = None if key is None else after_m.get(key, 0) - before_m.get(key, 0)
mark("local_inference", chunks=len(chunks), text_chars=len("".join(chunks)), counter=key, generated_delta=delta, **gpu_query())
verdict("local_inference", "PASS" if len(chunks) > 1 and "".join(chunks).strip() and (delta or 0) > 0 else "FAIL",
        f"{len(chunks)} chunks, generated-token counter +{delta}", chunks=len(chunks), generated_delta=delta)

rcheck("remote_inference_done", "local_inference_done", stage="inference", phase="after")

## 03-7a. Reviewed connector, runtime credential getter, stage helper

In [ ]:
# Fetch the exact reviewed Enigma connector revision (GitHub only; no Enigma request). Verifies the checked-out SHA.
REV = "8648d91632b5bbf5ae174403a01578c04a5eff20"
repo = f"{ATTEMPT_DIR}/enigma-src"
mark("connector_fetch_begin", rev=REV)
for cmd in (["git", "init", "-q", repo], ["git", "-C", repo, "fetch", "-q", "--depth", "1", "https://github.com/stokastisk-ai/enigma.git", REV],
            ["git", "-C", repo, "checkout", "-q", "FETCH_HEAD"]):
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    if r.returncode != 0:
        verdict("connector_fetch", "FAIL", f"{cmd[3] if len(cmd) > 3 else cmd[1]} failed rc={r.returncode}", stderr_tail=r.stderr[-500:])
head = subprocess.run(["git", "-C", repo, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
sys.path.insert(0, f"{repo}/connector")
import enigma_connector
from enigma_connector.supervise import ConnectorProcess            # stdlib-only module; run_stage is imported only where a stage runs
ok = head == REV and enigma_connector.__file__.startswith(repo)
verdict("connector_fetch", "PASS" if ok else "FAIL", f"HEAD={head}", head=head, module=enigma_connector.__file__)

In [ ]:
# Runtime credential (ENIGMA_RUNTIME_TOKEN): read through the shared helper only when a cloud stage needs it; type- and format-checked;
# never printed, stored, or put in os.environ. A diagnostic-key problem can never be reported as a runtime-token problem (separate secrets, codes, steps).
assert API_URL == MODES[INTEGRATION_MODE]["url"], "transport changed mid-attempt"     # selected by the capability gate; never switched
os.environ.pop("ENIGMA_RUNTIME_TOKEN", None)


def token_getter():
    t, res = get_credential("ENIGMA_RUNTIME_TOKEN")
    if t is None:
        raise ValueError(f"ENIGMA_RUNTIME_TOKEN {res['code']}: {res['message']}")
    return t

In [ ]:
from enigma_connector.stage_runner import run_stage, MarkerLog
from enigma_connector import serverctl
INTERVAL_ARGS = ("--gpu-interval", "1.0", "--vllm-interval", "2.0", "--upload-interval", "5.0")      # explicit finite intervals, as in every verified run
ORDER = ("nvml", "vllm", "register", "heartbeat", "jobs", "upload-once", "telemetry", "full")


def stage(name, **kw):
    # One SUPERVISED connector subprocess per stage (never in this kernel). Prerequisites (result AND durable evidence) must have PASSED in THIS session.
    prev = ORDER[:ORDER.index(name)]
    require("connector_fetch", "local_inference", *[f"stage_{p}" for p in prev], *[f"stage_{p}_evidence" for p in prev])
    if name not in ("nvml", "vllm"):
        require("cloud_gate")
    assert server.health(), "local vLLM unhealthy"
    cloud = name not in ("nvml", "vllm")
    # The before-marker MUST be durable (a failed one stops here). Offline stages send nothing DURING their measured interval.
    rcheck(f"stage_{name}_marker", f"before_stage_{name}", stage=name, phase="before")
    res = run_stage(name, api_url=API_URL if cloud else "", token_getter=token_getter if cloud else None, vllm_url=server.base_url, model=MODEL, tp=1,
                    server=server, workdir=f"{ATTEMPT_DIR}/stages", extra_args=INTERVAL_ARGS, markers=MarkerLog(f"{ATTEMPT_DIR}/markers.jsonl", echo=print), **kw)
    stats = (res.get("summary") or {}).get("stats") or {}
    extra_ok, why = True, ""
    if name == "jobs":
        extra_ok, why = stats.get("jobs_done") == 0 and stats.get("jobs_failed") == 0, "job stage must stay idle"
    if name == "full":
        extra_ok, why = (stats.get("jobs_done") or 0) >= 1 and stats.get("jobs_failed") == 0, "at least one real Playground job must complete"
    failed = [c[0] for c in res["checks"] if not c[1]] + ([why] if not extra_ok else [])
    status = "PASS" if res["passed"] and extra_ok else "FAIL"
    # The stage RESULT is recorded first and independently: a failed stage stays failed even if its marker is stored ...
    after = rmark(f"after_stage_{name}", stage=name, phase="after", exit_code=res["exit"]["returncode"], signal=res["exit"].get("killed_by_signal"), verdict=status)
    verdict(f"stage_{name}", status, f"rc={res['exit']['returncode']} failed={failed}", run_id=(res.get("summary") or {}).get("run_id"), stats=stats)
    # ... and a passed stage whose required durable evidence is missing can never satisfy the acceptance gate.
    verdict(f"stage_{name}_evidence", "PASS" if after else "INCONCLUSIVE", f"after-stage diagnostic marker {after.get('code')}")
    return res

## 03-7b. Offline stages (no Enigma request during the measured intervals)

In [ ]:
_ = stage("nvml", duration=120)

In [ ]:
_ = stage("vllm")

## 03-8. Cloud stages (selected endpoint only)

In [ ]:
# 03-7. Cloud gate (evidence, not a checkbox): the selected endpoint's capabilities and the REAL runtime identity were verified before any GPU work,
# the session survived, and both offline stages passed with durable evidence. A queued Playground job would be claimed by the idle-polling stage and
# fail it (it must stay idle): submit NOTHING before stage "full".
require("runtime_capabilities", "runtime_identity", "post_capability_survival", "stage_nvml", "stage_nvml_evidence", "stage_vllm", "stage_vllm_evidence")
assert API_URL == MODES[INTEGRATION_MODE]["url"], "transport changed mid-attempt"
rcheck("cloud_gate_marker", "cloud_gate_open", stage="cloud_gate", phase="before", target=RUNTIME_TRANSPORT, note=SCOPE)
verdict("cloud_gate", "PASS", f"{SCOPE} cloud stages enabled against {urllib.parse.urlsplit(API_URL).hostname} only")

In [ ]:
_ = stage("register")

In [ ]:
_ = stage("heartbeat")

In [ ]:
_ = stage("jobs")

In [ ]:
_ = stage("upload-once")

In [ ]:
_ = stage("telemetry")

## 03-8e. Full connector with one real Playground inference (600 s)
Start this cell, wait for `stage_full` to begin, then submit ONE prompt:

Open the Enigma web app on the OCI deployment (`<OCI_BASE_URL>/app/runtimes`), sign in, select the real runtime, open Playground and submit ONE prompt; confirm the streamed reply and the GPU charts in the UI.

PASS requires at least one completed job and no failed job.

In [ ]:
_ = stage("full", duration=600, poll_s=15)

## 03-9. Build-5-safe shutdown

In [ ]:
# 03-9. Shutdown with the build-5-safe orchestration from the reviewed connector (never patches the installed wheel).
rep = serverctl.safe_stop(server)
mark("server_safe_stop", report=rep, **gpu_query())
ok = rep.get("action") == "package_stop" and rep.get("report", {}).get("group_gone")
rmark("shutdown_done", stage="shutdown", phase="after", verdict="PASS" if ok else "FAIL", note=str(rep.get("action")))
verdict("server_shutdown", "PASS" if ok else "FAIL", f"{rep}")

## 03-10. Overall acceptance (run it last, also after a stop)

In [ ]:
# 03-10. OVERALL acceptance. Run it last, also after a stop (it reads this session's verdicts only). PASS needs EVERY step below, including safe
# shutdown and a durable final diagnostic verdict on the OCI deployment.
REQUIRED = ["environment", "diag_key_preflight", "runtime_token_preflight", "diag_endpoint", "baseline_manifest", "runtime_capabilities", "runtime_identity",
            "post_capability_survival", "wheel_verified", "install_plan", "install", "server_start", "local_inference", "connector_fetch", "cloud_gate",
            *[f"stage_{s}{e}" for s in ("nvml", "vllm", "register", "heartbeat", "jobs", "upload-once", "telemetry", "full") for e in ("", "_evidence")],
            "server_shutdown"]
got = {s: RESULTS.get(s, {}).get("status", "NOT_RUN") if RESULTS.get(s, {}).get("attempt") in (ATTEMPT, None) else "STALE" for s in REQUIRED}
if all(v == "PASS" for v in got.values()):
    outcome = "PASS"
elif any(RESULTS.get(s, {}).get("blocked") for s in ("runtime_capabilities", "diag_endpoint")):
    outcome = "BLOCKED"
elif any(v == "FAIL" for v in got.values()):
    outcome = "FAIL"
else:
    outcome = "INCONCLUSIVE"
fin = rmark("overall", stage="overall", verdict=outcome, note=f"{globals().get('SCOPE', INTEGRATION_MODE)}; missing={[k for k, v in got.items() if v != 'PASS'][:6]}")
if outcome == "PASS" and not fin:
    outcome = "INCONCLUSIVE"                          # the final remote verdict is required evidence
scope = globals().get("SCOPE", INTEGRATION_MODE.upper())
label = {"PASS": f"{scope} INTEGRATION ACCEPTED (OCI)"}.get(outcome, outcome)
RESULTS["overall"] = {"status": outcome, "reason": label, "attempt": ATTEMPT, "steps": got}
mark("verdict", step="overall", status=outcome, reason=label, steps=got, final_remote=fin.get("code"))
print(json.dumps(got, indent=1), flush=True)
print(f"VERDICT overall: {outcome} | {label}", flush=True)
if outcome != "PASS":
    raise DiagnosticStop(f"overall {outcome}")

## Evidence export (one compact, redacted archive)

In [ ]:
# Compact evidence export (run it even after a FAIL / INCONCLUSIVE / BLOCKED stop). ONE redacted archive directly in /kaggle/working (not inside the
# attempt tree, so it is never nested in another archive): markers, verdicts, pip reports, server and stage logs. The public 38 MB wheel bytes and the
# public connector clone are excluded (the wheel's size + SHA-256 and the connector's full SHA are in the markers). Download THIS file only; do not
# zip /kaggle/working/enigma-diag-attempts.
mark("export_begin", results={k: v["status"] for k, v in RESULTS.items()}, wheel_sha256="fe80d7c36fa78cf8a26feb1e00bb5f1f006ca45a027450353603976320ae0b3e", connector_rev="8648d91632b5bbf5ae174403a01578c04a5eff20")
archive = f"/kaggle/working/{NOTEBOOK}-{ATTEMPT}-evidence.zip"
files, hits = write_redacted_archive(ATTEMPT_DIR, archive)
print("evidence archive:", archive, os.path.getsize(archive), "bytes,", files, "files, redactions:", hits, flush=True)
print(json.dumps({k: {"status": v["status"], "reason": v["reason"]} for k, v in RESULTS.items()}, indent=1), flush=True)